1...Import Libraies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


2...Load Dataset

In [ ]:
df=pd.read_csv("Student Social Media And Mental Health Impact.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

3.. Exploratory Data Analysis

In [ ]:
sns.histplot(df['Mental_Health_Score'],kde=True)  #kde :kernel density estimater

In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=True)

In [ ]:
df['Stress_Level'].unique()

In [ ]:
df.info()

In [ ]:
sns.boxplot(x='Stress_Level',y='Mental_Health_Score',data=df)

In [ ]:
sns.scatterplot(x='Avg_Daily_Usage_Hours',y='Mental_Health_Score',data=df)

In [ ]:
sns.scatterplot(x="Sleep_Hours_Per_Night",y="Mental_Health_Score",data=df)

In [ ]:
sns.scatterplot(x="Study_Hours",y="Mental_Health_Score",data=df)

In [ ]:
df["Most_Used_Platform"].unique()

In [ ]:
platform_counts=df["Most_Used_Platform"].value_counts()
platform_counts

In [ ]:

plt.figure(figsize=(8,4))
sns.countplot(x=df["Most_Used_Platform"],order=df["Most_Used_Platform"].value_counts().index)

In [ ]:
#Checking outlier  : 
num_features=df.select_dtypes(include="number")
Q1=num_features.quantile(0.25)
Q3=num_features.quantile(0.75)
IQR=Q3- Q1

lower_bound = Q1 -1.5*IQR
upper_bound=Q3 + 1.5*IQR

outliers=(num_features< lower_bound)| (num_features >upper_bound)
print(outliers.sum())

4.. Data Cleaning 

In [ ]:
#1. To drop duplicates
df=df.drop_duplicates()

#2. converting value and unrealistic value to realistic value
df["Physical_Activity_Hours"]=df["Physical_Activity_Hours"].clip(lower=0)

5.. Skewness

In [ ]:
num_col= df.select_dtypes(include="number")
num_col.skew()

6..Feature Engineering

In [ ]:
top_countries=df["Country"].value_counts().index[:10].tolist()

In [ ]:
def group_countries(country):
    if country in top_countries:
        return country

    else:
        return 'Other'

In [ ]:
df["Grouped_Country"]=df["Country"].apply(group_countries)

In [ ]:
df["Grouped_Country"].value_counts()

7..Encoding Strategy

In [ ]:
df.columns

8..Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
skews_col=["Study_Hours"]  # log transformation
other_numeric_cols=["Age","Avg_Daily_Usage_Hours","Daily_Unlocks", "Physical_Activity_Hours", 
                    "Sleep_Hours_Per_Night" ]  #standard Scaling
ordinal_col=["Stress_Level"] 
normal_col=["Gender","Academic_Level","Most_Used_Platform","Purpose_Of_Use","Grouped_Country"]   #One hot encoding

feature_col=skews_col + other_numeric_cols+ ordinal_col + normal_col

X=df[feature_col]
y=df["Mental_Health_Score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

9..Preprocessing using ColumnTransformer

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
#1. Skewed feature 
skew_pipline =Pipeline(steps=[
    ('log_transform',FunctionTransformer(np.log1p)),
    ('scale',StandardScaler())
])

#2. Numeric Features
plain_numeric_pipeline=Pipeline(steps=[
    ('scale',StandardScaler())
])

#3.Ordinal 
ordinal_pipline=Pipeline(steps=[
    ('encode',OrdinalEncoder(categories=[['Low','Medium','High','Very High']]))
])

#Nominal Features
nominal_pipline=Pipeline(steps=[
    ('encode',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor=ColumnTransformer(transformers=[
    ("Skewed_Pipline",skew_pipline,skews_col),  #(pipline,feature)
    ("Plain_Numeric",plain_numeric_pipeline,other_numeric_cols),
    ("Odinal",ordinal_pipline,ordinal_col),
    ("Nominal",nominal_pipline,normal_col)
])


In [ ]:
preprocessor

10...Build A Pipline 

11...Model Building

In [ ]:
#Linear Regression 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

lr_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())

])
lr_pipeline.fit(X_train,y_train)

lr_preds=lr_pipeline.predict(X_test)

lr_preds_train=lr_pipeline.predict(X_train)
lr_mae=mean_absolute_error(y_test,lr_preds)

lr_r2_testing=r2_score(y_test,lr_preds)
lr_r2_training=r2_score(y_train,lr_preds_train)

print(f"Accuracy of Training :{lr_r2_training}")
print(f"Accuracy of Testing :{lr_r2_testing}")
print(f"MAE {lr_mae}")

In [ ]:
#Random Forest 
from sklearn.ensemble import RandomForestRegressor

rf_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('random forest',RandomForestRegressor(random_state=42))
])
rf_pipeline.fit(X_train,y_train)
rf_preds=rf_pipeline.predict(X_test)
rf_preds_training =rf_pipeline.predict(X_train)

rf_r2_testing=r2_score(y_test,rf_preds)
rf_r2_training=r2_score(y_train,rf_preds_training)
rf_mae= mean_absolute_error(y_test,rf_preds)

print(f"Accuracy of Training :{rf_r2_training}")
print(f"Accuracy of Testing :{rf_r2_testing}")
print(f"MAE :{rf_mae}")

In [ ]:
#Hyperparameter 
from sklearn.model_selection import RandomizedSearchCV
para_grid = {
    'random forest__n_estimators': [100, 200, 300],
    'random forest__max_depth': [5, 10, 15],
    'random forest__min_samples_split': [2, 5, 10],
    'random forest__min_samples_leaf': [1, 2, 4]
}

random_search=RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=para_grid,
    n_iter=15,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1   # use all free param to do hyperparameter

)
random_search.fit(X_train,y_train)


In [ ]:
random_search.best_params_

In [ ]:
rf_best_pipeline=random_search.best_estimator_
rf_best_preds=rf_best_pipeline.predict(X_test)

print(f"R2 score for random forest after hyperparameter tuning {r2_score(y_test,rf_best_preds)}")
print(f"MAE score for random forest after hyperparameter tuning {mean_absolute_error(y_test,rf_best_preds)}")

12... Model Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds          = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse           = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training       = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae            = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2             = r2_score(y_test, rf_tuned_preds)


# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)

13... Save the model  

In [ ]:
import joblib 
joblib.dump(rf_pipeline,'Mental_Health_Model.pkl')

In [ ]:
df.Stress_Level.unique().tolist()